# Chess World Model — Training Notebook

**Project:** En Parlant~ Chess Narrative AI
**Architecture:** Convolutional VAE (position encoder) + MDN-RNN (sequence model) + Sparse Autoencoder (concept extraction)
**Target Hardware:** NVIDIA RTX 5090 (32GB VRAM)
**Training Data:** En Parlant~ game databases (9M+ games)

This notebook trains a chess "world model" that learns strategic concepts unsupervised from millions of games. The trained model enables TTS narration of any chess game by mapping board positions to discovered strategic concepts.

## Architecture Overview

```
Board Position (8×8×20) → Conv VAE Encoder → z_t (256-dim latent)
                                                    ↓
Game Sequence [z_1..z_T] + [move_1..move_T] → MDN-RNN → h_t (512-dim context)
                                                              ↓
                                              Sparse Autoencoder → interpretable concepts
                                                              ↓
                                              Concept labels → TTS narration
```

## Phase 0: Environment Setup

In [ ]:
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

# Core dependencies
for pkg in ["torch", "chess", "numpy", "tqdm", "matplotlib", "tensorboard"]:
    try:
        __import__(pkg)
    except ImportError:
        install(f"python-{pkg}" if pkg == "chess" else pkg)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, IterableDataset
from torch.utils.tensorboard import SummaryWriter
import chess
import chess.pgn
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm
import io, sqlite3, struct, json, os, time
from collections import defaultdict
from dataclasses import dataclass

# ── GPU Setup ──
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    print("TF32 + cuDNN benchmark enabled")
else:
    print("WARNING: No GPU detected")

# ── Custom Blackwell Kernels (optional, sm_120 only) ──
BWK_AVAILABLE = False
BWK_PATH = "/data/src/bwk/fused-mlp"  # redshed path
try:
    sys.path.insert(0, f"{BWK_PATH}/python")
    from blackwell_kernels import fused_mlp_sm120, bf16_gemm
    BWK_AVAILABLE = True
    print("Blackwell kernels (bwk) loaded — fused_mlp_sm120, bf16_gemm available")
except (ImportError, OSError):
    print("Blackwell kernels not available, using PyTorch defaults")

print(f"\nPyTorch {torch.__version__} | Device: {device}")

## Phase 1: Configuration

In [ ]:
@dataclass
class Config:
    # ── Data ──
    db_paths: list = None  # populated below
    max_games: int = 2_000_000       # start with 2M, scale to 9M+
    max_ply: int = 200               # truncate games longer than this
    min_ply: int = 10                # skip very short games
    val_split: float = 0.02          # 2% validation

    # ── Position Encoder (V) ──
    input_channels: int = 20         # 12 pieces + 4 castling + 1 turn + 1 ep + 2 repetition
    encoder_channels: int = 128      # conv channel width
    encoder_blocks: int = 4          # residual blocks
    latent_dim: int = 256            # z_t dimension

    # ── Sequence Model (M) ──
    move_vocab_size: int = 1968      # all possible UCI moves (from×to×promo)
    move_embed_dim: int = 64         # move embedding dimension
    rnn_hidden_dim: int = 512        # h_t dimension
    rnn_layers: int = 2              # LSTM layers
    num_gaussians: int = 5           # MDN mixture components

    # ── Sparse Autoencoder (C) ──
    concept_dict_size: int = 2048    # overcomplete dictionary (4x hidden)
    sparsity_alpha: float = 1e-3     # L1 sparsity penalty

    # ── Training ──
    batch_size: int = 256
    seq_len: int = 60                # positions per game sample
    lr: float = 3e-4
    weight_decay: float = 1e-5
    kl_weight: float = 0.1           # VAE KL divergence weight (beta-VAE)
    kl_warmup_steps: int = 5000      # anneal KL from 0 to kl_weight
    epochs_vae: int = 10             # Phase 1: train V
    epochs_rnn: int = 10             # Phase 2: train M
    epochs_sae: int = 5              # Phase 3: train C
    grad_clip: float = 1.0
    checkpoint_dir: str = "/data/chess-world-model/checkpoints"
    log_dir: str = "/data/chess-world-model/logs"

cfg = Config()

# ── Populate database paths (redshed layout) ──
DB_BASE = "/data/chess-lab/database/db"
PGN_BASE = "/mnt/viperdrive/Storage_3/Chess_Data_Bases"

# En Parlant databases (already in .db3 format with encoded moves)
ep_dbs = list(Path(DB_BASE).glob("*.db3")) if Path(DB_BASE).exists() else []

# ChessBase PGN databases (raw PGN, need parsing)
cb_pgns = []
for ext in ["*.pgn", "*.pgn.zst", "*.pgn.bz2"]:
    cb_pgns.extend(Path(PGN_BASE).rglob(ext))

print(f"En Parlant databases: {len(ep_dbs)}")
for db in ep_dbs[:5]:
    print(f"  {db.name} ({db.stat().st_size / 1e9:.2f} GB)")
print(f"\nChessBase PGN files: {len(cb_pgns)}")
for pgn in cb_pgns[:5]:
    print(f"  {pgn.name}")

os.makedirs(cfg.checkpoint_dir, exist_ok=True)
os.makedirs(cfg.log_dir, exist_ok=True)

## Phase 2: Board Encoding — Position to Tensor

The AlphaZero input representation: 8×8 spatial grid with channels for each piece type, castling rights, side to move, en passant, and repetition count. This preserves the spatial structure so convolutional kernels can learn piece interactions (pins, forks, threats) naturally.

In [ ]:
# ── Piece channel mapping ──
# Channels 0-5:  White K, Q, R, B, N, P
# Channels 6-11: Black K, Q, R, B, N, P
# Channel 12:    Side to move (1=white, 0=black, all squares)
# Channel 13:    En passant square (1 on the target square)
# Channels 14-17: Castling rights (WK, WQ, BK, BQ, all squares)
# Channels 18-19: Repetition counters (1-fold, 2-fold, all squares)

PIECE_TO_CHANNEL = {
    (chess.KING, chess.WHITE): 0, (chess.QUEEN, chess.WHITE): 1,
    (chess.ROOK, chess.WHITE): 2, (chess.BISHOP, chess.WHITE): 3,
    (chess.KNIGHT, chess.WHITE): 4, (chess.PAWN, chess.WHITE): 5,
    (chess.KING, chess.BLACK): 6, (chess.QUEEN, chess.BLACK): 7,
    (chess.ROOK, chess.BLACK): 8, (chess.BISHOP, chess.BLACK): 9,
    (chess.KNIGHT, chess.BLACK): 10, (chess.PAWN, chess.BLACK): 11,
}

def board_to_tensor(board: chess.Board) -> np.ndarray:
    """Convert a python-chess Board to a 20×8×8 float32 tensor."""
    tensor = np.zeros((20, 8, 8), dtype=np.float32)

    # Piece placement (channels 0-11)
    for sq in chess.SQUARES:
        piece = board.piece_at(sq)
        if piece is not None:
            ch = PIECE_TO_CHANNEL[(piece.piece_type, piece.color)]
            rank, file = divmod(sq, 8)
            tensor[ch, rank, file] = 1.0

    # Side to move (channel 12)
    if board.turn == chess.WHITE:
        tensor[12, :, :] = 1.0

    # En passant (channel 13)
    if board.ep_square is not None:
        rank, file = divmod(board.ep_square, 8)
        tensor[13, rank, file] = 1.0

    # Castling rights (channels 14-17)
    if board.has_kingside_castling_rights(chess.WHITE):
        tensor[14, :, :] = 1.0
    if board.has_queenside_castling_rights(chess.WHITE):
        tensor[15, :, :] = 1.0
    if board.has_kingside_castling_rights(chess.BLACK):
        tensor[16, :, :] = 1.0
    if board.has_queenside_castling_rights(chess.BLACK):
        tensor[17, :, :] = 1.0

    # Repetition (channels 18-19) — set based on board state
    # We use is_repetition which checks the move stack
    if board.is_repetition(1):
        tensor[18, :, :] = 1.0
    if board.is_repetition(2):
        tensor[19, :, :] = 1.0

    return tensor


# ── Move tokenization ──
# UCI move string → integer token (e.g., "e2e4" → 423)
# There are at most 64*64 + promotions = ~4672 possible UCI strings,
# but only ~1968 are reachable in standard chess.

def build_move_vocab():
    """Build a vocabulary of all possible UCI move strings."""
    moves = []
    for from_sq in range(64):
        for to_sq in range(64):
            if from_sq == to_sq:
                continue
            uci = chess.square_name(from_sq) + chess.square_name(to_sq)
            moves.append(uci)
            # Promotions (only for pawn moves to rank 1 or 8)
            from_rank = from_sq // 8
            to_rank = to_sq // 8
            if (from_rank == 6 and to_rank == 7) or (from_rank == 1 and to_rank == 0):
                for promo in ["q", "r", "b", "n"]:
                    moves.append(uci + promo)
    vocab = {m: i for i, m in enumerate(moves)}
    return vocab

MOVE_VOCAB = build_move_vocab()
MOVE_PAD = len(MOVE_VOCAB)  # padding token
print(f"Move vocabulary size: {len(MOVE_VOCAB)} (+ 1 PAD = {MOVE_PAD + 1})")

def move_to_token(move: chess.Move) -> int:
    """Convert a chess.Move to a vocabulary index."""
    uci = move.uci()
    return MOVE_VOCAB.get(uci, MOVE_PAD)


# ── Verify encoding ──
board = chess.Board()
t = board_to_tensor(board)
print(f"\nInitial position tensor shape: {t.shape}")
print(f"White pawns (channel 5) rank 1: {t[5, 1, :]}")  # should be all 1s
print(f"Black pawns (channel 11) rank 6: {t[11, 6, :]}")  # should be all 1s
print(f"Side to move (white=1): {t[12, 0, 0]}")
print(f"White kingside castling: {t[14, 0, 0]}")

## Phase 3: Data Pipeline — PGN to Training Sequences

Two data sources:
1. **En Parlant .db3 databases** — games already encoded as move bytes, fast to decode
2. **Raw PGN files** — parsed with python-chess, slower but supports any PGN source

Each game becomes a sequence of (position_tensor, move_token) pairs.

In [ ]:
class PGNGameIterator:
    """Iterate over games from PGN files, yielding (positions, moves) sequences."""

    def __init__(self, pgn_paths, max_ply=200, min_ply=10):
        self.pgn_paths = pgn_paths
        self.max_ply = max_ply
        self.min_ply = min_ply

    def __iter__(self):
        for path in self.pgn_paths:
            try:
                opener = open
                if str(path).endswith(".zst"):
                    import zstandard as zstd
                    dctx = zstd.ZstdDecompressor()
                    fh = open(path, "rb")
                    opener = lambda p: io.TextIOWrapper(dctx.stream_reader(fh))
                elif str(path).endswith(".bz2"):
                    import bz2
                    opener = lambda p: bz2.open(p, "rt")

                with opener(path) as f:
                    while True:
                        game = chess.pgn.read_game(f)
                        if game is None:
                            break
                        result = self._process_game(game)
                        if result is not None:
                            yield result
            except Exception as e:
                print(f"Error reading {path}: {e}")
                continue

    def _process_game(self, game):
        board = game.board()
        positions = [board_to_tensor(board)]
        moves = []

        for move in game.mainline_moves():
            moves.append(move_to_token(move))
            board.push(move)
            positions.append(board_to_tensor(board))

            if len(moves) >= self.max_ply:
                break

        if len(moves) < self.min_ply:
            return None

        return {
            "positions": np.stack(positions),  # (T+1, 20, 8, 8)
            "moves": np.array(moves, dtype=np.int64),  # (T,)
        }


class ChessSequenceDataset(Dataset):
    """Pre-processed dataset of game sequences stored as numpy arrays."""

    def __init__(self, cache_dir, seq_len=60):
        self.cache_dir = Path(cache_dir)
        self.seq_len = seq_len
        self.index = []  # (file_idx, game_idx_in_file)

        # Load index
        idx_path = self.cache_dir / "index.json"
        if idx_path.exists():
            with open(idx_path) as f:
                self.index = json.load(f)
            print(f"Loaded cached dataset: {len(self.index)} game segments")
        else:
            print("No cached dataset found. Run build_dataset() first.")

    def __len__(self):
        return len(self.index)

    def __getitem__(self, idx):
        file_path, start_ply, length = self.index[idx]
        data = np.load(file_path, mmap_mode="r")

        # Extract sequence segment
        end = min(start_ply + self.seq_len + 1, start_ply + length + 1)
        actual_len = end - start_ply - 1

        positions = np.array(data["positions"][start_ply:end])  # (T+1, 20, 8, 8)
        moves = np.array(data["moves"][start_ply:end - 1])      # (T,)

        # Pad if needed
        T = self.seq_len
        if actual_len < T:
            pos_pad = np.zeros((T + 1 - positions.shape[0], 20, 8, 8), dtype=np.float32)
            positions = np.concatenate([positions, pos_pad])
            mov_pad = np.full((T - moves.shape[0],), MOVE_PAD, dtype=np.int64)
            moves = np.concatenate([moves, mov_pad])

        mask = np.zeros(T, dtype=np.float32)
        mask[:actual_len] = 1.0

        return {
            "positions": torch.from_numpy(positions),     # (T+1, 20, 8, 8)
            "moves": torch.from_numpy(moves),             # (T,)
            "mask": torch.from_numpy(mask),                # (T,)
            "length": actual_len,
        }


def build_dataset(pgn_paths, cache_dir, max_games=100_000, seq_len=60):
    """Process PGN files into cached numpy arrays for fast training."""
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)

    index = []
    game_count = 0
    batch_positions = []
    batch_moves = []
    batch_id = 0

    iterator = PGNGameIterator(pgn_paths)
    pbar = tqdm(iterator, desc="Processing games", total=max_games)

    for game_data in pbar:
        if game_count >= max_games:
            break

        positions = game_data["positions"]
        moves = game_data["moves"]
        T = len(moves)

        # Create overlapping segments of seq_len
        for start in range(0, T, seq_len // 2):
            length = min(seq_len, T - start)
            if length < 10:
                continue

            file_path = str(cache_dir / f"batch_{batch_id:06d}.npz")

            if len(batch_positions) == 0 or len(batch_positions[-1]["moves"]) != length:
                # Save current batch and start new one
                if batch_positions:
                    np.savez_compressed(
                        cache_dir / f"batch_{batch_id:06d}.npz",
                        positions=batch_positions[-1]["positions"],
                        moves=batch_positions[-1]["moves"],
                    )
                    batch_id += 1

            index.append((file_path, 0, length))

        # Simple approach: one npz per game
        file_path = cache_dir / f"game_{game_count:08d}.npz"
        np.savez_compressed(file_path, positions=positions, moves=moves)

        for start in range(0, T, seq_len // 2):
            length = min(seq_len, T - start)
            if length >= 10:
                index.append((str(file_path), start, length))

        game_count += 1
        pbar.set_postfix(games=game_count, segments=len(index))

    # Save index
    with open(cache_dir / "index.json", "w") as f:
        json.dump(index, f)

    print(f"\nDataset built: {game_count} games, {len(index)} training segments")
    print(f"Cache directory: {cache_dir}")
    return ChessSequenceDataset(cache_dir, seq_len)


# ── Quick test with a sample game ──
board = chess.Board()
sample_moves = ["e2e4", "e7e5", "g1f3", "b8c6", "f1b5"]
positions = [board_to_tensor(board)]
tokens = []
for uci in sample_moves:
    move = chess.Move.from_uci(uci)
    tokens.append(move_to_token(move))
    board.push(move)
    positions.append(board_to_tensor(board))

print(f"Sample game: {len(positions)} positions, {len(tokens)} moves")
print(f"Move tokens: {tokens}")
print(f"Position tensor dtype: {positions[0].dtype}, shape: {positions[0].shape}")

## Phase 4: Model Architecture

Three components trained in sequence:
1. **Position Encoder V** — Convolutional VAE that compresses 8×8×20 board states to 256-dim latent vectors
2. **Sequence Model M** — LSTM with Mixture Density Network output, predicts next position from game context
3. **Sparse Autoencoder C** — Post-hoc extraction of interpretable strategic concepts from M's hidden states

In [ ]:
# ═══════════════════════════════════════════════
# Component V: Convolutional VAE Position Encoder
# ═══════════════════════════════════════════════

class ResBlock(nn.Module):
    """Residual block with two 3×3 convolutions (AlphaZero-style)."""
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(channels)

    def forward(self, x):
        residual = x
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        return F.relu(x + residual)


class PositionEncoder(nn.Module):
    """Encodes an 8×8×20 board tensor to (mu, logvar) in latent space."""
    def __init__(self, cfg):
        super().__init__()
        C = cfg.encoder_channels

        # Initial projection: 20 → C channels
        self.input_conv = nn.Sequential(
            nn.Conv2d(cfg.input_channels, C, 3, padding=1, bias=False),
            nn.BatchNorm2d(C),
            nn.ReLU(),
        )

        # Residual tower
        self.res_blocks = nn.Sequential(*[ResBlock(C) for _ in range(cfg.encoder_blocks)])

        # Flatten 8×8×C → latent
        self.flatten_dim = C * 8 * 8
        self.fc_mu = nn.Linear(self.flatten_dim, cfg.latent_dim)
        self.fc_logvar = nn.Linear(self.flatten_dim, cfg.latent_dim)

    def forward(self, x):
        # x: (B, 20, 8, 8)
        x = self.input_conv(x)
        x = self.res_blocks(x)
        x = x.flatten(1)  # (B, C*64)
        return self.fc_mu(x), self.fc_logvar(x)


class PositionDecoder(nn.Module):
    """Decodes a latent vector back to 8×8×20 board tensor."""
    def __init__(self, cfg):
        super().__init__()
        C = cfg.encoder_channels

        self.fc = nn.Linear(cfg.latent_dim, C * 8 * 8)
        self.reshape_channels = C

        self.res_blocks = nn.Sequential(*[ResBlock(C) for _ in range(cfg.encoder_blocks)])

        self.output_conv = nn.Sequential(
            nn.Conv2d(C, cfg.input_channels, 3, padding=1),
            # No final activation — reconstruction targets are binary,
            # loss uses BCE with logits
        )

    def forward(self, z):
        x = self.fc(z)
        x = x.view(-1, self.reshape_channels, 8, 8)
        x = self.res_blocks(x)
        return self.output_conv(x)  # (B, 20, 8, 8) logits


class ChessVAE(nn.Module):
    """Variational Autoencoder for chess positions."""
    def __init__(self, cfg):
        super().__init__()
        self.encoder = PositionEncoder(cfg)
        self.decoder = PositionDecoder(cfg)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        mu, logvar = self.encoder(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decoder(z)
        return recon, mu, logvar, z

    def encode(self, x):
        mu, logvar = self.encoder(x)
        return mu  # use mean for deterministic encoding

    @staticmethod
    def loss(recon, target, mu, logvar, kl_weight=0.1):
        # Reconstruction: BCE with logits (target is binary)
        recon_loss = F.binary_cross_entropy_with_logits(recon, target, reduction="mean")
        # KL divergence
        kl_loss = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
        return recon_loss + kl_weight * kl_loss, recon_loss, kl_loss


# ── Verify model ──
vae = ChessVAE(cfg)
sample_input = torch.randn(4, 20, 8, 8)
recon, mu, logvar, z = vae(sample_input)
print(f"VAE parameter count: {sum(p.numel() for p in vae.parameters()):,}")
print(f"Input: {sample_input.shape} → Latent: {z.shape} → Recon: {recon.shape}")

In [ ]:
# ═══════════════════════════════════════════════
# Component M: MDN-RNN Sequence Model
# ═══════════════════════════════════════════════

class MDNRNN(nn.Module):
    """LSTM + Mixture Density Network for predicting next latent position.

    Input:  (z_t, a_t) at each timestep — latent position + move token
    Output: Parameters of a Gaussian mixture for z_{t+1}
    Hidden: h_t accumulates strategic context (this is what we extract)
    """

    def __init__(self, cfg):
        super().__init__()
        self.latent_dim = cfg.latent_dim
        self.hidden_dim = cfg.rnn_hidden_dim
        self.num_gaussians = cfg.num_gaussians

        # Move embedding
        self.move_embed = nn.Embedding(cfg.move_vocab_size + 1, cfg.move_embed_dim,
                                       padding_idx=MOVE_PAD)

        # LSTM: input = z_t (256) + move_embed (64) = 320
        input_dim = cfg.latent_dim + cfg.move_embed_dim
        self.lstm = nn.LSTM(input_dim, cfg.rnn_hidden_dim,
                           num_layers=cfg.rnn_layers,
                           batch_first=True, dropout=0.1)

        # MDN output: for each Gaussian, predict (mu, sigma, pi) for each latent dim
        # pi: mixing weights (num_gaussians)
        # mu: means (num_gaussians × latent_dim)
        # sigma: std devs (num_gaussians × latent_dim)
        n_out = cfg.num_gaussians * (1 + 2 * cfg.latent_dim)
        self.mdn_head = nn.Linear(cfg.rnn_hidden_dim, n_out)

    def forward(self, z_seq, move_seq, hidden=None):
        """
        Args:
            z_seq: (B, T, latent_dim) — encoded positions
            move_seq: (B, T) — move tokens
            hidden: optional initial LSTM state

        Returns:
            pi, mu, sigma: MDN parameters for z_{t+1}
            hidden: final LSTM hidden state
        """
        B, T, _ = z_seq.shape

        move_emb = self.move_embed(move_seq)  # (B, T, move_embed_dim)
        rnn_input = torch.cat([z_seq, move_emb], dim=-1)  # (B, T, 320)

        rnn_out, hidden = self.lstm(rnn_input, hidden)  # (B, T, hidden_dim)

        mdn_out = self.mdn_head(rnn_out)  # (B, T, n_out)

        return self._parse_mdn(mdn_out), hidden, rnn_out

    def _parse_mdn(self, mdn_out):
        """Parse MDN output into (pi, mu, sigma)."""
        B, T, _ = mdn_out.shape
        K = self.num_gaussians
        D = self.latent_dim

        # Split output
        pi_logits = mdn_out[:, :, :K]                          # (B, T, K)
        mu = mdn_out[:, :, K:K + K*D].view(B, T, K, D)        # (B, T, K, D)
        log_sigma = mdn_out[:, :, K + K*D:].view(B, T, K, D)  # (B, T, K, D)

        pi = F.softmax(pi_logits, dim=-1)                       # (B, T, K)
        sigma = torch.exp(log_sigma).clamp(min=1e-6, max=10.0)  # (B, T, K, D)

        return pi, mu, sigma

    @staticmethod
    def mdn_loss(pi, mu, sigma, target, mask=None):
        """Negative log-likelihood of target under the mixture model.

        Args:
            pi: (B, T, K) mixing weights
            mu: (B, T, K, D) means
            sigma: (B, T, K, D) std devs
            target: (B, T, D) target z_{t+1}
            mask: (B, T) binary mask for valid timesteps
        """
        target = target.unsqueeze(2)  # (B, T, 1, D)

        # Log probability under each Gaussian
        var = sigma ** 2
        log_prob = -0.5 * (
            torch.log(2 * np.pi * var) + (target - mu) ** 2 / var
        ).sum(dim=-1)  # (B, T, K)

        # Log mixture probability
        log_mix_prob = torch.log(pi + 1e-10) + log_prob  # (B, T, K)
        log_likelihood = torch.logsumexp(log_mix_prob, dim=-1)  # (B, T)

        if mask is not None:
            log_likelihood = log_likelihood * mask
            return -log_likelihood.sum() / mask.sum().clamp(min=1)
        return -log_likelihood.mean()

    def get_context(self, z_seq, move_seq, hidden=None):
        """Get the strategic context vectors h_t (for concept extraction)."""
        move_emb = self.move_embed(move_seq)
        rnn_input = torch.cat([z_seq, move_emb], dim=-1)
        rnn_out, hidden = self.lstm(rnn_input, hidden)
        return rnn_out, hidden  # rnn_out is h_t at each timestep


# ── Verify ──
rnn = MDNRNN(cfg)
z_seq = torch.randn(4, 30, cfg.latent_dim)
m_seq = torch.randint(0, 100, (4, 30))
(pi, mu, sigma), hidden, h_t = rnn(z_seq, m_seq)
print(f"MDN-RNN parameter count: {sum(p.numel() for p in rnn.parameters()):,}")
print(f"Input z: {z_seq.shape}, moves: {m_seq.shape}")
print(f"Output pi: {pi.shape}, mu: {mu.shape}, sigma: {sigma.shape}")
print(f"Context h_t: {h_t.shape}")

In [ ]:
# ═══════════════════════════════════════════════
# Component C: Sparse Autoencoder for Concept Extraction
# ═══════════════════════════════════════════════
# Applied AFTER training V and M. Decomposes h_t vectors into
# interpretable strategic concepts (Bricken et al., 2023).

class SparseAutoencoder(nn.Module):
    """Sparse autoencoder for extracting monosemantic features from h_t.

    Learns an overcomplete dictionary of strategic concepts.
    Each concept activates sparsely — ideally one concept = one chess idea.
    """
    def __init__(self, input_dim, dict_size, tied_weights=True):
        super().__init__()
        self.input_dim = input_dim
        self.dict_size = dict_size

        # Encoder: h_t → sparse concept activations
        self.encoder = nn.Linear(input_dim, dict_size, bias=True)

        if tied_weights:
            # Tied weights: decoder = encoder^T (Bricken et al. recommend this)
            self.decoder_weight = None  # use encoder weight transposed
        else:
            self.decoder = nn.Linear(dict_size, input_dim, bias=False)
            self.decoder_weight = self.decoder.weight

        self.tied_weights = tied_weights

        # Normalize encoder rows (dictionary features) to unit norm
        with torch.no_grad():
            self.encoder.weight.data = F.normalize(self.encoder.weight.data, dim=1)

    def forward(self, x):
        # Encode: sparse concept activations
        c = F.relu(self.encoder(x))  # (B, dict_size) — sparse via ReLU

        # Decode: reconstruct h_t
        if self.tied_weights:
            x_hat = F.linear(c, self.encoder.weight.t())
        else:
            x_hat = self.decoder(c)

        return x_hat, c

    def loss(self, x, alpha=1e-3):
        x_hat, c = self.forward(x)
        recon_loss = F.mse_loss(x_hat, x)
        sparsity_loss = c.abs().mean()  # L1 on activations
        return recon_loss + alpha * sparsity_loss, recon_loss, sparsity_loss, c

    def get_top_concepts(self, x, k=10):
        """Get the top-k active concepts for a given h_t vector."""
        with torch.no_grad():
            c = F.relu(self.encoder(x))
            values, indices = c.topk(k, dim=-1)
        return indices, values


# ── Verify ──
sae = SparseAutoencoder(cfg.rnn_hidden_dim, cfg.concept_dict_size)
h_sample = torch.randn(32, cfg.rnn_hidden_dim)
x_hat, c = sae(h_sample)
print(f"SAE parameter count: {sum(p.numel() for p in sae.parameters()):,}")
print(f"Input h_t: {h_sample.shape} → Concepts: {c.shape} → Recon: {x_hat.shape}")
print(f"Sparsity: {(c > 0).float().mean():.3f} (fraction of non-zero activations)")
print(f"Active concepts per sample: {(c > 0).sum(dim=-1).float().mean():.1f} / {cfg.concept_dict_size}")

## Phase 5: Training Loops

Three-phase training:
1. **Train VAE** on individual positions (reconstruct board states, learn z_t)
2. **Train MDN-RNN** on game sequences using frozen VAE encoder (predict z_{t+1} from z_t + move)
3. **Train Sparse Autoencoder** on h_t vectors from frozen MDN-RNN (extract concepts)

In [ ]:
# ═══════════════════════════════════════════════
# Training Phase 1: VAE Position Encoder
# ═══════════════════════════════════════════════

def train_vae(vae, dataloader, val_loader, cfg, writer=None):
    """Train the VAE to reconstruct chess positions from latent vectors."""
    vae = vae.to(device)
    optimizer = torch.optim.AdamW(vae.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, cfg.epochs_vae * len(dataloader))

    global_step = 0
    best_val_loss = float("inf")

    for epoch in range(cfg.epochs_vae):
        vae.train()
        epoch_loss = 0
        epoch_recon = 0
        epoch_kl = 0

        pbar = tqdm(dataloader, desc=f"VAE Epoch {epoch+1}/{cfg.epochs_vae}")
        for batch in pbar:
            positions = batch["positions"].to(device)  # (B, T+1, 20, 8, 8)
            B, T1, C, H, W = positions.shape

            # Flatten all positions into one batch for the VAE
            x = positions.view(B * T1, C, H, W)

            # KL warmup: anneal from 0 to kl_weight
            kl_w = min(cfg.kl_weight, cfg.kl_weight * global_step / max(cfg.kl_warmup_steps, 1))

            recon, mu, logvar, z = vae(x)
            loss, recon_loss, kl_loss = ChessVAE.loss(recon, x, mu, logvar, kl_w)

            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(vae.parameters(), cfg.grad_clip)
            optimizer.step()
            scheduler.step()

            epoch_loss += loss.item()
            epoch_recon += recon_loss.item()
            epoch_kl += kl_loss.item()
            global_step += 1

            pbar.set_postfix(loss=f"{loss.item():.4f}", recon=f"{recon_loss.item():.4f}",
                           kl=f"{kl_loss.item():.4f}", kl_w=f"{kl_w:.4f}")

            if writer and global_step % 100 == 0:
                writer.add_scalar("vae/loss", loss.item(), global_step)
                writer.add_scalar("vae/recon_loss", recon_loss.item(), global_step)
                writer.add_scalar("vae/kl_loss", kl_loss.item(), global_step)
                writer.add_scalar("vae/kl_weight", kl_w, global_step)
                writer.add_scalar("vae/lr", scheduler.get_last_lr()[0], global_step)

        # Validation
        val_loss = evaluate_vae(vae, val_loader, cfg)
        n_batches = len(dataloader)
        print(f"  Epoch {epoch+1}: train_loss={epoch_loss/n_batches:.4f} "
              f"recon={epoch_recon/n_batches:.4f} kl={epoch_kl/n_batches:.4f} "
              f"val_loss={val_loss:.4f}")

        if writer:
            writer.add_scalar("vae/val_loss", val_loss, epoch)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(vae.state_dict(), f"{cfg.checkpoint_dir}/vae_best.pt")
            print(f"  Saved best VAE (val_loss={val_loss:.4f})")

    return vae


@torch.no_grad()
def evaluate_vae(vae, dataloader, cfg):
    vae.eval()
    total_loss = 0
    n = 0
    for batch in dataloader:
        positions = batch["positions"].to(device)
        B, T1, C, H, W = positions.shape
        x = positions.view(B * T1, C, H, W)
        recon, mu, logvar, z = vae(x)
        loss, _, _ = ChessVAE.loss(recon, x, mu, logvar, cfg.kl_weight)
        total_loss += loss.item()
        n += 1
    return total_loss / max(n, 1)


print("VAE training function defined.")

In [ ]:
# ═══════════════════════════════════════════════
# Training Phase 2: MDN-RNN Sequence Model
# ═══════════════════════════════════════════════

def train_rnn(rnn, vae, dataloader, val_loader, cfg, writer=None):
    """Train the MDN-RNN to predict next latent position from game context."""
    vae = vae.to(device).eval()  # freeze VAE
    rnn = rnn.to(device)
    optimizer = torch.optim.AdamW(rnn.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, cfg.epochs_rnn * len(dataloader))

    global_step = 0
    best_val_loss = float("inf")

    for epoch in range(cfg.epochs_rnn):
        rnn.train()
        epoch_loss = 0

        pbar = tqdm(dataloader, desc=f"RNN Epoch {epoch+1}/{cfg.epochs_rnn}")
        for batch in pbar:
            positions = batch["positions"].to(device)  # (B, T+1, 20, 8, 8)
            moves = batch["moves"].to(device)          # (B, T)
            mask = batch["mask"].to(device)             # (B, T)
            B, T1, C, H, W = positions.shape
            T = T1 - 1

            # Encode all positions with frozen VAE
            with torch.no_grad():
                all_pos = positions.view(B * T1, C, H, W)
                z_all = vae.encode(all_pos)  # (B*T1, latent_dim)
                z_all = z_all.view(B, T1, -1)  # (B, T+1, latent_dim)

            z_input = z_all[:, :-1, :]   # (B, T, D) — positions 0..T-1
            z_target = z_all[:, 1:, :]   # (B, T, D) — positions 1..T

            (pi, mu, sigma), hidden, h_t = rnn(z_input, moves)

            # Predict z_{t+1} from (z_t, a_t)
            # mask out padding
            loss = MDNRNN.mdn_loss(pi, mu, sigma, z_target, mask)

            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(rnn.parameters(), cfg.grad_clip)
            optimizer.step()
            scheduler.step()

            epoch_loss += loss.item()
            global_step += 1
            pbar.set_postfix(loss=f"{loss.item():.4f}")

            if writer and global_step % 100 == 0:
                writer.add_scalar("rnn/loss", loss.item(), global_step)
                writer.add_scalar("rnn/lr", scheduler.get_last_lr()[0], global_step)

        # Validation
        val_loss = evaluate_rnn(rnn, vae, val_loader, cfg)
        n_batches = len(dataloader)
        print(f"  Epoch {epoch+1}: train_loss={epoch_loss/n_batches:.4f} val_loss={val_loss:.4f}")

        if writer:
            writer.add_scalar("rnn/val_loss", val_loss, epoch)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(rnn.state_dict(), f"{cfg.checkpoint_dir}/rnn_best.pt")
            print(f"  Saved best RNN (val_loss={val_loss:.4f})")

    return rnn


@torch.no_grad()
def evaluate_rnn(rnn, vae, dataloader, cfg):
    rnn.eval()
    total_loss = 0
    n = 0
    for batch in dataloader:
        positions = batch["positions"].to(device)
        moves = batch["moves"].to(device)
        mask = batch["mask"].to(device)
        B, T1, C, H, W = positions.shape

        all_pos = positions.view(B * T1, C, H, W)
        z_all = vae.encode(all_pos).view(B, T1, -1)
        z_input = z_all[:, :-1, :]
        z_target = z_all[:, 1:, :]

        (pi, mu, sigma), _, _ = rnn(z_input, moves)
        loss = MDNRNN.mdn_loss(pi, mu, sigma, z_target, mask)
        total_loss += loss.item()
        n += 1
    return total_loss / max(n, 1)


print("MDN-RNN training function defined.")

In [ ]:
# ═══════════════════════════════════════════════
# Training Phase 3: Sparse Autoencoder (Concept Extraction)
# ═══════════════════════════════════════════════

@torch.no_grad()
def collect_hidden_states(rnn, vae, dataloader, max_samples=1_000_000):
    """Collect h_t vectors from the trained RNN for SAE training."""
    vae.eval()
    rnn.eval()
    all_h = []
    total = 0

    for batch in tqdm(dataloader, desc="Collecting h_t vectors"):
        positions = batch["positions"].to(device)
        moves = batch["moves"].to(device)
        mask = batch["mask"].to(device)
        B, T1, C, H, W = positions.shape

        all_pos = positions.view(B * T1, C, H, W)
        z_all = vae.encode(all_pos).view(B, T1, -1)
        z_input = z_all[:, :-1, :]

        h_t, _ = rnn.get_context(z_input, moves)  # (B, T, hidden_dim)

        # Flatten and filter by mask
        h_flat = h_t.view(-1, h_t.shape[-1])  # (B*T, hidden_dim)
        m_flat = mask.view(-1)
        valid_h = h_flat[m_flat > 0]

        all_h.append(valid_h.cpu())
        total += valid_h.shape[0]

        if total >= max_samples:
            break

    all_h = torch.cat(all_h, dim=0)[:max_samples]
    print(f"Collected {all_h.shape[0]:,} h_t vectors, shape: {all_h.shape}")
    return all_h


def train_sae(sae, h_data, cfg, writer=None):
    """Train the sparse autoencoder on collected h_t vectors."""
    sae = sae.to(device)
    optimizer = torch.optim.Adam(sae.parameters(), lr=1e-3)

    dataset = torch.utils.data.TensorDataset(h_data)
    loader = DataLoader(dataset, batch_size=4096, shuffle=True, num_workers=4)

    global_step = 0
    for epoch in range(cfg.epochs_sae):
        sae.train()
        epoch_loss = 0
        epoch_recon = 0
        epoch_sparse = 0

        for (batch_h,) in tqdm(loader, desc=f"SAE Epoch {epoch+1}/{cfg.epochs_sae}"):
            batch_h = batch_h.to(device)

            loss, recon_loss, sparsity_loss, c = sae.loss(batch_h, cfg.sparsity_alpha)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # Re-normalize dictionary features to unit norm (important!)
            with torch.no_grad():
                sae.encoder.weight.data = F.normalize(sae.encoder.weight.data, dim=1)

            epoch_loss += loss.item()
            epoch_recon += recon_loss.item()
            epoch_sparse += sparsity_loss.item()
            global_step += 1

            if writer and global_step % 50 == 0:
                writer.add_scalar("sae/loss", loss.item(), global_step)
                writer.add_scalar("sae/recon_loss", recon_loss.item(), global_step)
                writer.add_scalar("sae/sparsity_loss", sparsity_loss.item(), global_step)
                active_frac = (c > 0).float().mean().item()
                writer.add_scalar("sae/active_fraction", active_frac, global_step)

        n = len(loader)
        print(f"  Epoch {epoch+1}: loss={epoch_loss/n:.4f} recon={epoch_recon/n:.4f} "
              f"sparsity={epoch_sparse/n:.4f}")

    torch.save(sae.state_dict(), f"{cfg.checkpoint_dir}/sae_best.pt")
    print("Saved SAE checkpoint.")
    return sae


print("SAE training function defined.")

## Phase 6: Run Training

Execute the full three-phase pipeline. Adjust `cfg.max_games` for prototyping (start small, scale up).

In [ ]:
# ═══════════════════════════════════════════════
# BUILD DATASET (run once, then cached)
# ═══════════════════════════════════════════════
# Uncomment and run to build the training data cache:

# pgn_paths = list(Path("/data/chess-lab/database").rglob("*.pgn"))
# dataset = build_dataset(pgn_paths, "/data/chess-world-model/cache", max_games=cfg.max_games)

# For prototyping, use a smaller sample:
# pgn_paths = ["/data/chess-lab/database/sample.pgn"]
# dataset = build_dataset(pgn_paths, "/data/chess-world-model/cache_small", max_games=1000)

In [ ]:
# ═══════════════════════════════════════════════
# RUN FULL TRAINING PIPELINE
# ═══════════════════════════════════════════════

def run_training(cfg):
    """Execute the complete three-phase training pipeline."""
    print("=" * 60)
    print("CHESS WORLD MODEL — FULL TRAINING PIPELINE")
    print("=" * 60)

    # Load cached dataset
    cache_dir = "/data/chess-world-model/cache"
    dataset = ChessSequenceDataset(cache_dir, cfg.seq_len)

    if len(dataset) == 0:
        print("ERROR: No cached dataset found. Run build_dataset() first.")
        return

    # Train/val split
    val_size = int(len(dataset) * cfg.val_split)
    train_size = len(dataset) - val_size
    train_ds, val_ds = torch.utils.data.random_split(dataset, [train_size, val_size])

    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,
                             num_workers=8, pin_memory=True, persistent_workers=True)
    val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False,
                           num_workers=4, pin_memory=True)

    print(f"Training: {train_size:,} segments | Validation: {val_size:,} segments")
    print(f"Batch size: {cfg.batch_size} | Sequence length: {cfg.seq_len}")

    writer = SummaryWriter(cfg.log_dir)

    # ── Phase 1: Train VAE ──
    print("\n" + "=" * 60)
    print("PHASE 1: Training Position Encoder (VAE)")
    print("=" * 60)
    vae = ChessVAE(cfg)
    vae = train_vae(vae, train_loader, val_loader, cfg, writer)

    # ── Phase 2: Train MDN-RNN ──
    print("\n" + "=" * 60)
    print("PHASE 2: Training Sequence Model (MDN-RNN)")
    print("=" * 60)
    rnn = MDNRNN(cfg)
    rnn = train_rnn(rnn, vae, train_loader, val_loader, cfg, writer)

    # ── Phase 3: Train Sparse Autoencoder ──
    print("\n" + "=" * 60)
    print("PHASE 3: Training Concept Extractor (Sparse Autoencoder)")
    print("=" * 60)
    h_data = collect_hidden_states(rnn, vae, train_loader)
    sae = SparseAutoencoder(cfg.rnn_hidden_dim, cfg.concept_dict_size)
    sae = train_sae(sae, h_data, cfg, writer)

    writer.close()

    # ── Summary ──
    total_params = (sum(p.numel() for p in vae.parameters()) +
                   sum(p.numel() for p in rnn.parameters()) +
                   sum(p.numel() for p in sae.parameters()))
    print("\n" + "=" * 60)
    print("TRAINING COMPLETE")
    print(f"Total parameters: {total_params:,}")
    print(f"Checkpoints: {cfg.checkpoint_dir}")
    print(f"TensorBoard logs: {cfg.log_dir}")
    print("=" * 60)

    return vae, rnn, sae

# ── Uncomment to run ──
# vae, rnn, sae = run_training(cfg)

## Phase 7: Concept Exploration

After training, inspect what the sparse autoencoder learned. Each dictionary feature should correspond to an interpretable chess concept.

In [ ]:
# ═══════════════════════════════════════════════
# CONCEPT EXPLORATION & VISUALIZATION
# ═══════════════════════════════════════════════

@torch.no_grad()
def explore_concepts(sae, rnn, vae, dataloader, top_k=20, concepts_to_show=10):
    """Find the chess positions that most strongly activate each concept."""
    vae.eval(); rnn.eval(); sae.eval()

    # Track top-activating positions for each concept
    concept_activations = defaultdict(list)  # concept_id → [(activation, fen, move_number)]

    for batch in tqdm(dataloader, desc="Scanning for concept activations"):
        positions = batch["positions"].to(device)
        moves = batch["moves"].to(device)
        mask = batch["mask"].to(device)
        B, T1, C, H, W = positions.shape

        all_pos = positions.view(B * T1, C, H, W)
        z_all = vae.encode(all_pos).view(B, T1, -1)
        z_input = z_all[:, :-1, :]

        h_t, _ = rnn.get_context(z_input, moves)  # (B, T, 512)

        # Get concept activations
        for b in range(B):
            for t in range(int(mask[b].sum().item())):
                h = h_t[b, t].unsqueeze(0)
                _, c = sae(h)
                top_vals, top_ids = c[0].topk(5)

                # Reconstruct FEN from position tensor (approximate)
                pos_tensor = positions[b, t].cpu().numpy()

                for val, cid in zip(top_vals, top_ids):
                    cid = cid.item()
                    concept_activations[cid].append({
                        "activation": val.item(),
                        "move_number": t,
                        "position_hash": hash(pos_tensor.tobytes()),
                    })

    # Sort by activation strength and show top concepts
    print("\n" + "=" * 60)
    print("TOP CONCEPT ACTIVATIONS")
    print("=" * 60)

    concept_stats = []
    for cid, activations in sorted(concept_activations.items(),
                                    key=lambda x: -max(a["activation"] for a in x[1])):
        activations.sort(key=lambda x: -x["activation"])
        avg_move = np.mean([a["move_number"] for a in activations[:top_k]])
        max_act = activations[0]["activation"]
        count = len(activations)

        concept_stats.append({
            "concept_id": cid,
            "max_activation": max_act,
            "count": count,
            "avg_move_number": avg_move,
        })

    # Show top concepts
    for i, cs in enumerate(concept_stats[:concepts_to_show]):
        phase = "opening" if cs["avg_move_number"] < 15 else \
                "middlegame" if cs["avg_move_number"] < 35 else "endgame"
        print(f"\nConcept #{cs['concept_id']:4d}: "
              f"max_act={cs['max_activation']:.3f}, "
              f"appears_in={cs['count']:,} positions, "
              f"avg_move={cs['avg_move_number']:.1f} ({phase})")

    return concept_stats


# ── Uncomment after training ──
# concept_stats = explore_concepts(sae, rnn, vae, val_loader)

print("Concept exploration function defined.")

## Phase 8: ONNX Export

Export trained models to ONNX for integration with En Parlant~ (which already bundles ONNX Runtime for KittenTTS).

In [ ]:
# ═══════════════════════════════════════════════
# ONNX EXPORT
# ═══════════════════════════════════════════════

def export_to_onnx(vae, rnn, sae, cfg, output_dir="/data/chess-world-model/onnx"):
    """Export all three models to ONNX format for deployment in En Parlant~."""
    os.makedirs(output_dir, exist_ok=True)
    vae.eval().cpu(); rnn.eval().cpu(); sae.eval().cpu()

    # Export VAE encoder only (we don't need the decoder at inference time)
    print("Exporting VAE encoder...")
    dummy_pos = torch.randn(1, 20, 8, 8)
    torch.onnx.export(
        vae.encoder, dummy_pos,
        f"{output_dir}/chess_position_encoder.onnx",
        input_names=["board_tensor"],
        output_names=["mu", "logvar"],
        dynamic_axes={"board_tensor": {0: "batch"}},
        opset_version=17,
    )
    encoder_size = os.path.getsize(f"{output_dir}/chess_position_encoder.onnx") / 1e6
    print(f"  → chess_position_encoder.onnx ({encoder_size:.1f} MB)")

    # Export RNN (tricky with LSTM state — export as script module)
    print("Exporting sequence model...")
    # For ONNX, we export a wrapper that takes z + move and returns h_t
    class RNNWrapper(nn.Module):
        def __init__(self, rnn):
            super().__init__()
            self.rnn = rnn
        def forward(self, z_seq, move_seq):
            h_t, _ = self.rnn.get_context(z_seq, move_seq)
            return h_t

    rnn_wrapper = RNNWrapper(rnn)
    dummy_z = torch.randn(1, 10, cfg.latent_dim)
    dummy_m = torch.randint(0, 100, (1, 10))
    torch.onnx.export(
        rnn_wrapper, (dummy_z, dummy_m),
        f"{output_dir}/chess_sequence_model.onnx",
        input_names=["latent_positions", "moves"],
        output_names=["context_vectors"],
        dynamic_axes={
            "latent_positions": {0: "batch", 1: "seq_len"},
            "moves": {0: "batch", 1: "seq_len"},
        },
        opset_version=17,
    )
    rnn_size = os.path.getsize(f"{output_dir}/chess_sequence_model.onnx") / 1e6
    print(f"  → chess_sequence_model.onnx ({rnn_size:.1f} MB)")

    # Export SAE
    print("Exporting concept extractor...")
    dummy_h = torch.randn(1, cfg.rnn_hidden_dim)
    torch.onnx.export(
        sae, dummy_h,
        f"{output_dir}/chess_concept_extractor.onnx",
        input_names=["context_vector"],
        output_names=["reconstruction", "concept_activations"],
        dynamic_axes={"context_vector": {0: "batch"}},
        opset_version=17,
    )
    sae_size = os.path.getsize(f"{output_dir}/chess_concept_extractor.onnx") / 1e6
    print(f"  → chess_concept_extractor.onnx ({sae_size:.1f} MB)")

    total = encoder_size + rnn_size + sae_size
    print(f"\nTotal ONNX model size: {total:.1f} MB")
    print("Ready to bundle with En Parlant~!")


# ── Uncomment after training ──
# export_to_onnx(vae, rnn, sae, cfg)

print("ONNX export function defined.")